In [2]:
import polars as pl
import altair as alt
from pathlib import Path


In [3]:
import jellyfish
def clean_occurrences(entries):
    actual_occurrences = ["Suicide (method)", "Suicide (attempt)", "Homicide", "Homicide Attempt", "Escape", "Escape Attempt",
                "Fire", "Serious Injury", "Battery", "Riot of Rebellion", "Sex Offense", "Assault on Staff",
                "Assault among Detainees", "Fighting among Detainees", "Restraints Used", "OC Spray Used", "Other (specify)"]
    
    if entries.is_empty():
        return "Error, no occurrence found"

    cleaned_list = []
    for term in entries:
        # Skip None or empty terms
        if term is None or str(term).strip() == "":
            continue
        
        best_choice = None
        highest_jaro = 0.0
        for compare_term in actual_occurrences:
            jaro_score = jellyfish.jaro_similarity(str(term), compare_term)  # Convert to str to be safe
            if jaro_score > highest_jaro:
                highest_jaro = jaro_score
                best_choice = compare_term
        
        # Add with minimum threshold if found a good match
        if best_choice is not None and highest_jaro > 0.5:
            if best_choice in ["Suicide (method)", "Suicide (attempt)", "Other (specify)"]:
                # Keep the original term (possibly has colon text, like "Other (specify): Fight")
                cleaned_list.append(str(term))
            else:
                if best_choice not in cleaned_list:
                    cleaned_list.append(best_choice)
        else:
            # If no good match found, keep original term but clean it
            cleaned_term = str(term).strip()
            if cleaned_term and cleaned_term not in cleaned_list:
                cleaned_list.append(cleaned_term)
    
    return "; ".join(cleaned_list) if cleaned_list else "No occurrence found"

### **Merging datasets** 

- Tuesday and Friday datasets

In [8]:
root = Path('..')
#parquet_path = root / 'data/jails-data/output/jails_pdfs.parquet'
new_parquet_path = root / 'data/jails-data/SERVER/new run/output/jails_pds_cleanned.parquet'
old_parquet_path = root / 'data/jails-data/SERVER/old run/output/jails_pdfs_full.parquet'
df_old = pl.read_parquet(old_parquet_path)
df_new = pl.read_parquet(new_parquet_path)

# Rename the new occurrence columns 
df_new_renamed = df_new.with_columns(
    pl.col("Occurrence").alias("Occurrence_new")
).drop("Occurrence")

df_old_occurrence = df_old.select(["Report ID", "Occurrence"]).with_columns(
    pl.col("Occurrence").alias("Occurrence_old")
)

df_merged = df_new_renamed.join(
    df_old_occurrence,
    on="Report ID",
    how="left"  # Keep all records from new dataset
)

df_final = df_merged.with_columns([
    # Clean the old occurrence column
    pl.col("Occurrence_old").map_elements(clean_occurrences, return_dtype=pl.String).alias("Cleaned_Occurrence_Old"),
    # Clean the new occurrence column  
    pl.col("Occurrence_new").map_elements(clean_occurrences, return_dtype=pl.String).alias("Cleaned_Occurrence_New")
])

print("Columns in final dataset:")
print([col for col in df_final.columns if 'occurrence' in col.lower() or 'Occurrence' in col])

# Show a sample to compare
print("\nSample comparison of occurrence columns:")
sample_comparison = df_final.select([
    "Report ID", 
    "Occurrence_old", 
    "Cleaned_Occurrence_Old",
    "Occurrence_new", 
    "Cleaned_Occurrence_New"
])

sample_comparison

Columns in final dataset:
['Cleaned Occurrences', 'Occurrence_new', 'Occurrence', 'Occurrence_old', 'Cleaned_Occurrence_Old', 'Cleaned_Occurrence_New']

Sample comparison of occurrence columns:


Report ID,Occurrence_old,Cleaned_Occurrence_Old,Occurrence_new,Cleaned_Occurrence_New
str,list[str],str,list[str],str
"""Kiger, Bonham 1-2_p1""","[""Restraints Used ""]","""Restraints Used""","[""Restraints Used —[ ""]","""Restraints Used"""
"""Mateos, Gustavo 1-14_p1""","[""Other (specify): Resisted ""]","""Other (specify): Resisted ""","[""Other (specify): Resisted ""]","""Other (specify): Resisted """
"""Sinclair, Wendell 1-18_p1""","[""DC Spray Used "", ""ther (specify): Disorderly C ""]","""OC Spray Used; ther (specify):…","[""DC Spray Used "", ""ther (specify): Disorderly C ""]","""OC Spray Used; ther (specify):…"
"""Searle, Robert 1-21_p1""","[""Serious Injury ""]","""Serious Injury""","[""Serious Iniury ""]","""Serious Injury"""
"""Sutton, Micah 1-6_p1""","[""Restraints Used ""]","""Restraints Used""","[""Restraints Used ""]","""Restraints Used"""
…,…,…,…,…
"""UO - FOIA January 2023_p611""","[""Other (specify): STATEMENTS ""]","""Other (specify): STATEMENTS ""","[""Other (specify): STATEMENTS ""]","""Other (specify): STATEMENTS """
"""UO - FOIA January 2023_p612""","[""OC Spray Used "", ""ther (specify): Disorderly ""]","""OC Spray Used; ther (specify):…","[""OC Spray Used "", ""Dther (specify): Disorderly ""]","""OC Spray Used; Dther (specify)…"
"""UO - FOIA January 2023_p613""","[""Assault on Staff ""]","""Assault on Staff""","[""Assault on Staff ""]","""Assault on Staff"""


In [6]:
df_final

Deceased Name,Deceased Cause,Deceased Date and Time,Deceased on Suicide Watch,Deceased Reporter,Deceased Examined by Physician,Deceased Signs of Illness,Facility Type,RD Number,Facility Name,Phone Number,Address,Date,Time of Day,AM or PM,Table Contents,Injuries?,Resulting Death?,process,Report ID,OCR_Confidence,OCR_Word_Count,OCR_Text_Detected,Cook County?,Cleaned Facility Name,Cleaned Address,Zip Code,Cleaned Date,Cleaned Time of Day,Cleaned Occurrences,Occurrence_new,Occurrence,Occurrence_old,Cleaned_Occurrence_Old,Cleaned_Occurrence_New
str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],str,str,str,str,f64,i64,bool,str,str,str,str,str,str,str,list[str],list[str],list[str],str,str
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Error creating ROI: zero-lengt…","""R.D. Number: ; ""","""-acility Name: St. Clair Count…",""": 618-277-3505""","""\ddress: 700 North 5"" Street B…","""Date of Occurrence: 01/02/2021…","""me of Occurrence: 11:10 Oam. ""","""Error: less than two boxes fou…","[""| 12/21/1995"", ""| 01/02/2021"", … ""Kiger, Bonham""]","""No injuries""","""No ""","""normal""","""Kiger, Bonham 1-2_p1""",87.941718,326,true,null,"""St Clair County Sheriffs Depar…","""700 North 5"" Street Belleville…","""62220""","""01/02/2021""","""11:10 Oam.""","""Restraints Used""","[""Restraints Used —[ ""]","[""Restraints Used ""]","[""Restraints Used ""]","""Restraints Used""","""Restraints Used"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""Error: less than two boxes fou…","""R.D. Number: 21-00542 | ""","""Facility Name: Lake County Jai…",""": 847-377-4099""","""Address: 29 § Mlk Jr Ave Wauke…","""Date of Occurrence: 01/14/2021…","""_ Time of Occurrence: 12:23Pm ""","""p.m. ""","[""posse es"", ""_ Mateos, Gustavo"", … ""Robbery""]","""No injuries""","""Error: less than two boxes fou…","""normal""","""Mateos, Gustavo 1-14_p1""",88.17757,321,true,null,"""Lake County Jail""","""29 § Mlk Jr Ave Waukegan JL 60…","""60085""","""01/14/2021""","""12:23Pm""","""Resisted""","[""Other (specify): Resisted ""]","[""Other (specify): Resisted ""]","[""Other (specify): Resisted ""]","""Other (specify): Resisted ""","""Other (specify): Resisted """
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""R.D. Number: 21-00689 ""","""-acility Name: Lake County Jai…",""": 847-377-4099""","""\ddress: 29 S. Martin Luther K…","""Date of Occurrence: 01/18/2021…","""_ Time of Occurrence: 11:10 ""","""a.m, ""","[""Sinclair, Wendell"", ""06/ 17/ 1996"", … ""| Retail then / Disp Merch""]","""No injuries""","""Error creating ROI: zero-lengt…","""normal""","""Sinclair, Wendell 1-18_p1""",89.354633,313,true,null,"""Lake County Jail""","""29 S. Martin Luther King Jr. A…","""60085""","""01/18/2021""","""11:10""","""Disorderly C""","[""DC Spray Used "", ""ther (specify): Disorderly C ""]","[""DC Spray Used "", ""ther (specify): Disorderly C ""]","[""DC Spray Used "", ""ther (specify): Disorderly C ""]","""OC Spray Used; ther (specify):…","""OC Spray Used; ther (specify):…"
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""",""">| ""","""R.D.Number: ""","""Fax: (217) 522-3906 Facility N…",""": 309-888-4628""","""Facility Name: McLean County J…","""Aqaress: 1U4 West rront sucer …","""pre AN EEN et City State Time …","""p.m. ""","[""| METHAMPHETAMINE"", ""“12h 17/2020"", … ""| _TRAFFICKING _""]","""No injuries""","""No ""","""normal""","""Searle, Robert 1-21_p1""",88.421538,325,true,null,"""Mclean County Jail""","""Facility Name: McLean County J…",null,null,"""FLO""","""Serious Injury""","[""Serious Iniury ""]","[""Serious Injury ""]","[""Serious Injury ""]","""Serious Injury""","""Serious Injury"""
"""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""N/A""","""County ""","""R.D. Number: | ""","""Facility Name: Madison County …",""": 618-692-1065""","""Address: 405 Randle Street Edw…","""Date of Occurrence: 01/06/2021…","""_ Time of Occurrenc